In [39]:
import cv2
#from inference.models.utils import get_roboflow_model
import supervision as sv
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path



In [40]:
#model = get_roboflow_model(model_id="yolov8n-640") 
model = YOLO("yolov8n.pt")

In [6]:
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
trace_annotator = sv.TraceAnnotator(thickness=3,trace_length=30) # crea el efecto estela


In [7]:
tracker = sv.ByteTrack()


In [8]:
NARANJA_BAJO = np.array([5, 100, 100])
NARANJA_ALTO = np.array([25, 255, 255])


In [10]:
def comprobar_si_es_naranja(crop_objeto):
    """Analiza si el objeto recortado es predominantemente naranja"""
    if crop_objeto.size == 0:
        return False
    hsv_crop = cv2.cvtColor(crop_objeto, cv2.COLOR_BGR2HSV)
    mascara = cv2.inRange(hsv_crop, NARANJA_BAJO, NARANJA_ALTO)
    proporcion_naranja = np.sum(mascara == 255) / mascara.size
    return proporcion_naranja > 0.15 # Si más del 15% de los píxeles son naranjas


In [11]:
def callback(frame: np.ndarray, index: int) -> np.ndarray:
    # Ejecutar inferencia con Roboflow
    resultados = model.infer(frame)[0]
    
    # Convertir resultados al formato unificado de Supervision
    detecciones = sv.Detections.from_inference(resultados)
    
    # FILTRO 1: Quedarse únicamente con la clase 'sports ball' (ID 32 en COCO)
    # Si usas tu propio modelo entrenado de Roboflow, cambia el 32 por el ID de tu clase (suele ser 0)
    detecciones = detecciones[detecciones.class_id == 32]
    
    # FILTRO 2: Validar el color de las pelotas detectadas antes de dibujarlas
    indices_validos = []
    for i, bbox in enumerate(detecciones.xyxy):
        x1, y1, x2, y2 = map(int, bbox)
        recorte_pelota = frame[y1:y2, x1:x2] # Extraer la pelota de la imagen
        
        if comprobar_si_es_naranja(recorte_pelota):
            indices_validos.append(i)
            
    # Conservar únicamente las detecciones que pasaron el filtro de color naranja
    detecciones = detecciones[indices_validos]
    
    # 4. Aplicar rastreo (asigna un ID único a la pelota para seguir su trayectoria)
    detecciones = tracker.update_with_detections(detecciones)
    
    # 5. Dibujar los elementos visuales sobre el frame
    anotado = frame.copy()
    anotado = trace_annotator.annotate(scene=anotado, detections=detecciones)
    anotado = box_annotator.annotate(scene=anotado, detections=detecciones)
    
    # Crear etiquetas dinámicas ("Pelota Naranja" + ID de rastreo)
    etiquetas = [f"Pelota Naranja #{track_id}" for track_id in detecciones.tracker_id]
    anotado = label_annotator.annotate(scene=anotado, detections=detecciones, labels=etiquetas)
    
    return anotado


In [12]:
sv.process_video(
    source_path="assets/futbot-2s.mp4",
    target_path="resultado_output.mp4", # Generará adicionalmente un video guardado
    callback=callback
)

AttributeError: 'DetectionModel' object has no attribute 'infer'

In [49]:
cap = cv2.VideoCapture(r"assets/IMG_9933.MOV.mp4")

frame_width= int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(f'dimenciones: {frame_width}x{frame_height}')

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(r'assets/video.avi', fourcc,fps,(frame_width,frame_height))

dimenciones: 1080x1920


In [55]:
print(cap.isOpened()) 
while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break

    # 4. Ejecutar YOLO en el cuadro actual
    results = model(frame, verbose=False, conf=0.25)
    
    for result in results:
        for box in result.boxes:
            clase= int(box.cls)
            confianza = float(box.conf)
            
            if clase == 32:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                
                cv2.rectangle(frame, (x1,y1), (x2,y2), (0, 127, 255), 3)
                #cv2.putText(frame, f"Pelota: {confianza:.2f}", (int(x1),int(y1, -10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,127,255),2)
    out.write(frame)
    cv2.imshow('deteccion',frame)
cap.release()
out.release()
cv2.destroyAllWindows()


True


In [ ]:
# el video ya se puede encontrar en la ruta especifica
print(cap.isOpened()) 
while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break

    # 4. Ejecutar YOLO en el cuadro actual
    results = model(frame, verbose=False)[0]

    for box in results.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]

        # Comprobar si el objeto detectado es una pelota ('sports ball')
        if label == "sports ball":
            print("pelota encontrada")
            # Obtener las coordenadas de la caja de detección
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Recortar la región de la pelota (Region of Interest)
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            # 5. Analizar el color de la región recortada
            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            mask = cv2.inRange(hsv_roi,NARANJA_BAJO , NARANJA_ALTO)
            
            # Contar cuántos píxeles en la caja son verdaderamente naranjas
            orange_pixel_count = cv2.countNonZero(mask)
            total_pixels = roi.shape[0] * roi.shape[1]
            
            # Si más del 15% de la caja es naranja, la clasificamos correctamente
            if (orange_pixel_count / total_pixels) > 0.15:
                # Dibujar la caja y la etiqueta en el video
                print('Es una pelota noranja')
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 3)
                cv2.putText(frame, "Pelota Naranja", (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    # Mostrar el resultado en tiempo real
    #cv2.imwrite("assets/img.png", frame)
    #extencion.write(frame)
    #extencion.release()
    out.write(frame)
    cv2.imshow('detectando',frame)
    # Presiona 'q' para salir del video antes de que termine
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


True


AttributeError: 'list' object has no attribute 'boxes'